# Chapter 15 — Generative Models (intro)

Runnable companion to `docs/14_generative_models_intro.md`. We train a
tiny **VAE** on MNIST (the demo), then sketch **GAN** and **diffusion**
without a full training run — the point is the *concept*, not a
benchmark.

Generated by `src/build_chapter_14_vae_gan_diffusion_intro.py`.

## Setup

In [1]:
import math
from pathlib import Path as _Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device("cpu")

_ROOT = _Path.cwd()
while not (_ROOT / "ROADMAP.md").exists() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
MNIST_ROOT = str(_ROOT / "datasets" / "mnist")
print("torch:", torch.__version__, "device:", DEVICE)

torch: 2.12.0+cu130 device: cpu


## A tiny VAE on MNIST

The VAE is the simplest of the three. Encoder outputs `(μ, log σ²)`;
we sample `z = μ + σ · ε`; decoder reconstructs `x̂`. Loss is BCE
reconstruction + KL divergence to `N(0, I)`.

```
L = BCE(x̂, x) + 0.5 * Σ ( exp(log σ²) + μ² − 1 − log σ² )
```

In [2]:
class TinyVAE(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.ReLU(inplace=True),
        )
        self.fc_mu     = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 28 * 28),
            nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = (0.5 * logvar).exp()
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def vae_loss(x_hat, x, mu, logvar):
    recon = F.binary_cross_entropy(x_hat, x, reduction="sum")
    kl    = 0.5 * (logvar.exp() + mu.pow(2) - 1 - logvar).sum()
    return (recon + kl) / x.size(0), recon.item() / x.size(0), kl.item() / x.size(0)

In [3]:
transform = transforms.Compose([transforms.ToTensor()])
mnist_train = datasets.MNIST(MNIST_ROOT, train=True,  download=True, transform=transform)
mnist_val   = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=transform)
train_loader = DataLoader(Subset(mnist_train, list(range(4000))), batch_size=128, shuffle=True)
val_loader   = DataLoader(Subset(mnist_val,   list(range(1000))), batch_size=128)

## Train the VAE (5 epochs)

In [4]:
torch.manual_seed(0)
vae = TinyVAE(latent_dim=16).to(DEVICE)
opt = torch.optim.Adam(vae.parameters(), lr=1e-3)

history = []
for epoch in range(5):
    vae.train()
    running, recon_sum, kl_sum, n = 0.0, 0.0, 0.0, 0
    for x, _ in train_loader:
        x = x.to(DEVICE)
        opt.zero_grad()
        x_hat, mu, logvar = vae(x)
        loss, r, k = vae_loss(x_hat, x, mu, logvar)
        loss.backward(); opt.step()
        running += loss.item() * x.size(0); n += x.size(0)
        recon_sum += r * x.size(0); kl_sum += k * x.size(0)
    history.append((running / n, recon_sum / n, kl_sum / n))
    print(f"  epoch {epoch + 1}: loss={running/n:.2f} (recon={recon_sum/n:.2f}, kl={kl_sum/n:.2f})")

/home/bangbc/miniforge3/envs/aicourse/lib/python3.11/site-packages/torch/autograd/graph.py:882: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


  epoch 1: loss=330.75 (recon=317.39, kl=13.36)


  epoch 2: loss=217.05 (recon=209.16, kl=7.89)


  epoch 3: loss=203.12 (recon=195.17, kl=7.95)


  epoch 4: loss=187.51 (recon=177.12, kl=10.39)


  epoch 5: loss=175.75 (recon=163.81, kl=11.94)


## Inspect what the VAE learned

Three checks, in order of how cheaply they reveal a broken model:

1. **Reconstructions of real digits.** Does `decoder(encoder(x))` look
   like `x`?
2. **Samples from the prior.** Does `decoder(z ~ N(0, I))` look like
   *plausible* digits?
3. **Latent walk between two digits.** Does the model interpolate
   smoothly along the latent axis?

In [5]:
vae.eval()
with torch.no_grad():
    x_batch, _ = next(iter(val_loader))
    x_batch = x_batch[:8].to(DEVICE)
    x_hat, _, _ = vae(x_batch)

fig, axes = plt.subplots(2, 8, figsize=(11, 3))
for i in range(8):
    axes[0, i].imshow(x_batch[i, 0].cpu(), cmap="gray"); axes[0, i].axis("off")
    axes[1, i].imshow(x_hat[i, 0].cpu(),   cmap="gray"); axes[1, i].axis("off")
axes[0, 0].set_title("original",   loc="left", fontsize=9)
axes[1, 0].set_title("reconstruct", loc="left", fontsize=9)
plt.tight_layout(); plt.show()

/tmp/ipykernel_510332/1698879063.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [6]:
# Samples from the prior.
torch.manual_seed(0)
with torch.no_grad():
    z = torch.randn(8, vae.latent_dim, device=DEVICE)
    samples = vae.decode(z)

fig, axes = plt.subplots(1, 8, figsize=(11, 1.7))
for i in range(8):
    axes[i].imshow(samples[i, 0].cpu(), cmap="gray"); axes[i].axis("off")
plt.suptitle("Samples from z ~ N(0, I)", y=1.2)
plt.tight_layout(); plt.show()

/tmp/ipykernel_510332/3167290453.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [7]:
# Latent walk between two encoded digits.
with torch.no_grad():
    x_a = x_batch[0:1]; x_b = x_batch[3:4]
    mu_a, _ = vae.encode(x_a); mu_b, _ = vae.encode(x_b)
    walk = []
    for alpha in torch.linspace(0, 1, 8):
        z = (1 - alpha) * mu_a + alpha * mu_b
        walk.append(vae.decode(z))
    walk = torch.cat(walk, dim=0)

fig, axes = plt.subplots(1, 8, figsize=(11, 1.7))
for i in range(8):
    axes[i].imshow(walk[i, 0].cpu(), cmap="gray"); axes[i].axis("off")
plt.suptitle("Latent walk: digit A -> digit B", y=1.2)
plt.tight_layout(); plt.show()

/tmp/ipykernel_510332/1433502114.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## GAN — concept sketch (not trained)

We build the generator + discriminator but only show one *untrained*
forward pass + one discriminator step. Full GAN training is sensitive
to many hyperparameters and out of scope for the chapter.

In [8]:
class TinyGenerator(nn.Module):
    def __init__(self, z_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z_dim, 256), nn.ReLU(inplace=True),
            nn.Linear(256, 28 * 28), nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class TinyDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256), nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return self.net(x)


G = TinyGenerator(z_dim=16); D = TinyDiscriminator()
torch.manual_seed(0)
fake = G(torch.randn(8, 16))
d_score_fake = torch.sigmoid(D(fake))
d_score_real = torch.sigmoid(D(x_batch))
print(f"D(fake)  ~ {d_score_fake.mean().item():.3f}  (untrained: ~0.5)")
print(f"D(real)  ~ {d_score_real.mean().item():.3f}  (untrained: ~0.5)")
print("After training: D(fake) -> 0, D(real) -> 1; G then learns to push D(fake) -> 1.")

D(fake)  ~ 0.531  (untrained: ~0.5)
D(real)  ~ 0.524  (untrained: ~0.5)
After training: D(fake) -> 0, D(real) -> 1; G then learns to push D(fake) -> 1.


## Diffusion — concept sketch (forward process only)

The forward process gradually corrupts a real image with Gaussian
noise. The model is trained to *reverse* that — predicting the noise
that was added at each step. We visualize the forward process to make
the idea concrete.

In [9]:
T = 6              # number of noise steps to visualize
beta = torch.linspace(0.0001, 0.1, T)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

img = x_batch[0:1]
fig, axes = plt.subplots(1, T + 1, figsize=(11, 1.7))
axes[0].imshow(img[0, 0].cpu(), cmap="gray"); axes[0].axis("off")
axes[0].set_title("x_0", fontsize=8)
torch.manual_seed(0)
for t in range(T):
    eps = torch.randn_like(img)
    a_bar_t = alpha_bar[t]
    x_t = (a_bar_t.sqrt() * img + (1 - a_bar_t).sqrt() * eps)
    axes[t + 1].imshow(x_t[0, 0].cpu().clamp(-1, 2), cmap="gray"); axes[t + 1].axis("off")
    axes[t + 1].set_title(f"x_{t+1}", fontsize=8)
plt.suptitle("Forward diffusion: x_0 -> ... -> pure noise", y=1.2)
plt.tight_layout(); plt.show()

/tmp/ipykernel_510332/2322497136.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


The forward process is just additive Gaussian noise with a known
schedule — no learning. The *reverse* process is learned: at each step
the model predicts the noise `eps` that was added, and sampling
subtracts it back step by step from pure noise to recover an image.

## Comparison cheat-sheet

| Family    | What is learned          | Sampling step count | Strength            |
|-----------|--------------------------|--------------------:|---------------------|
| VAE       | encoder + decoder         | 1                   | stable, interpretable latent |
| GAN       | generator + discriminator | 1                   | sharp samples       |
| Diffusion | denoising network         | tens to hundreds    | state-of-the-art quality |

Pick by application. For pedagogy, VAE is the cleanest.